# Accessing [BioModels](https://www.ebi.ac.uk/biomodels/) with BioServices

**BioModels** is a repository of mathematical models of biological and
biomedical systems.  Models are encoded in standard formats such as SBML.

This notebook shows how to:

- Search and browse available models
- Retrieve model metadata
- Download model files

> **BioModels REST API**: https://www.ebi.ac.uk/biomodels/


In [4]:
import matplotlib.pyplot as plt
import pandas as pd
from bioservices import BioModels

s = BioModels()


INFO    [bioservices.BioModels:354]:  Initialising BioModels service (REST)


## 1. Searching for models

Use `search` to find models by keyword.  The result contains model metadata
such as identifier, name and number of reactions/species.


In [5]:
# Search for models related to MAPK signalling
results = s.search("MAPK")
print(type(results))
if isinstance(results, dict):
    models = results.get("models", [])
    print(f"Found {len(models)} models matching 'MAPK'")
    for m in models[:5]:
        print(f"  {m.get('id')}: {m.get('name')}")
else:
    print(results)


<class 'dict'>
Found 10 models matching 'MAPK'
  BIOMD0000000441: Sarma2012 - Oscillations in MAPK cascade (S2)
  BIOMD0000000440: Sarma2012 - Oscillations in MAPK cascade (S1)
  BIOMD0000000443: Sarma2012 - Oscillations in MAPK cascade (S1n)
  BIOMD0000000444: Sarma2012 - Oscillations in MAPK cascade (S2n)
  BIOMD0000000659: Cursons2015 - Regulation of ERK-MAPK signaling in human epidermis


## 2. Retrieving model metadata

Use `get_model` to fetch detailed metadata for a specific model by its
BioModels identifier (e.g. `MODEL1305240000`).


In [6]:
# Retrieve metadata for a specific model
biomodel = s.get_model("BIOMD0000000001")
print("Model ID   :", biomodel.get("publicationId") or biomodel.get("id", "n/a"))
print("Model name :", biomodel.get("name"))
print("Authors    :", biomodel.get("submitter", "n/a"))
desc = biomodel.get("description", "")
print("Description (first 200 chars):", str(desc)[:200])


Model ID   : BIOMD0000000001
Model name : Edelstein1996 - EPSP ACh event
Authors    : n/a
Description (first 200 chars): <notes xmlns="http://www.sbml.org/sbml/level2/version4">      <body xmlns="http://www.w3.org/1999/xhtml">        <div class="dc:title">Edelstein1996 - EPSP ACh event</div><div class="dc:description"> 


## 3. Listing model files

A BioModels entry may include several associated files (SBML, MATLAB scripts,
data files, etc.).


In [7]:
# List files available for model BIOMD0000000001
files = s.get_model_files("BIOMD0000000001")
if isinstance(files, dict):
    for category, entries in files.items():
        print(f"[{category}]")
        if isinstance(entries, list):
            for e in entries:
                print(f"  {e.get('name', e)}")
        else:
            print(f"  {entries}")
else:
    print(files)


[additional]
  BIOMD0000000001-biopax2.owl
  BIOMD0000000001-biopax3.owl
  BIOMD0000000001-matlab.m
  BIOMD0000000001-octave.m
  BIOMD0000000001.m
  BIOMD0000000001.ode
  BIOMD0000000001.pdf
  BIOMD0000000001.png
  BIOMD0000000001.svg
  BIOMD0000000001.vcml
  BIOMD0000000001_url.sedml
  curation_image.png
  curation_notes.txt
  manifest.xml
  metadata.rdf
[main]
  BIOMD0000000001_url.xml


## 4. Downloading a model file

Use `get_model_download` to save the SBML file to disk.


In [8]:
import tempfile, os

# Download the SBML file to a temporary location
with tempfile.TemporaryDirectory() as tmpdir:
    dest = os.path.join(tmpdir, "BIOMD0000000001.xml")
    s.get_model_download("BIOMD0000000001", "BIOMD0000000001.xml", output_filename=dest)
    size = os.path.getsize(dest)
    print(f"Downloaded SBML file: {dest} ({size} bytes)")
    with open(dest) as fh:
        content = fh.read()
    print("First 400 characters:")
    print(content[:400])


WARNING [bioservices.BioModels:599]:  status is not ok with Bad Request
INFO    [bioservices.BioModels:165]:  Saving BIOMD0000000001.xml


AttributeError: 'int' object has no attribute 'content'

## 5. Browsing the model list

Retrieve a batch of curated models and inspect their properties.


In [9]:
# Get the first page of curated models
all_models = s.search("*", numResults=20)
if isinstance(all_models, dict):
    model_list = all_models.get("models", [])
    df = pd.DataFrame([
        {"id": m.get("id"), "name": m.get("name"), "format": m.get("format", "SBML")}
        for m in model_list
    ])
    print(f"Loaded {len(df)} models")
    print(df.head(10).to_string(index=False))


Loaded 20 models
             id                                                                                 name format
MODEL1507180061 Sohn2012 - Genome-scale metabolic network of Schizosaccharomyces pombe (SpoMBEL1693)   SBML
BIOMD0000000595         Capuani2015 - Binding of Cbl and Grb2 to EGFR (Early Activation Model - EAM)   SBML
MODEL0848062679                                                      Irvine1999_CardiacSodiumChannel   SBML
MODEL1507180036      Pinchuck2010 - Genome-scale metabolic network of Shewanella oneidensis (iSO783)   SBML
BIOMD0000000963                        Weitz2020 - SIR model of COVID-19 transmission with shielding   SBML
MODEL7743358405                                               Radulescu2008_NFkB_hierarchy_M_8_12_19   SBML
MODEL1507180012    Förster2008 - Genome-scale metabolic network of Saccharamyces cerevisiae (iFF708)   SBML
MODEL8236480549                                                                Yang2008_AAnetwork_EC   SBML
MODEL091127

---
For more information, please see the
[bioservices.biomodels module documentation](https://bioservices.readthedocs.io/en/main/references.html#module-bioservices.biomodels)
and the [BioModels website](https://www.ebi.ac.uk/biomodels/).
